In [6]:
# WEEK 8: SCALED DOT-PRODUCT ATTENTION
# AND MULTI-HEAD ATTENTION

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# Scaled Dot-Product Attention

def scaled_dot_product_attention(Q, K, V, mask=None):

    d_k = Q.size(-1)

    # Calculate similarity scores
    scores = torch.matmul(
        Q, K.transpose(-2, -1)
    ) / math.sqrt(d_k)

    # Apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(
            mask == 0, float('-inf')
        )

    # Apply Softmax
    attention_weights = F.softmax(
        scores, dim=-1
    )

    # Calculate attention output
    output = torch.matmul(
        attention_weights, V
    )

    return output, attention_weights


# Multi-Head Attention Class

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model: int, num_heads: int):

        super().__init__()

        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear layers for Query, Key and Value
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Final output layer
        self.W_o = nn.Linear(d_model, d_model)

    # Split into multiple heads
    def split_heads(self, x):

        batch_size, seq_len, _ = x.size()

        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_k
        )

        return x.transpose(1, 2)

    # Combine all heads
    def combine_heads(self, x):

        batch_size, _, seq_len, _ = x.size()

        x = x.transpose(1, 2).contiguous()

        return x.view(
            batch_size,
            seq_len,
            self.d_model
        )

    # Forward function
    def forward(self, query, key, value, mask=None):

        # Generate Q, K and V
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # Split into multiple heads
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # Apply Scaled Dot-Product Attention
        attn_output, attn_weights = scaled_dot_product_attention(
            Q, K, V, mask
        )

        # Combine attention heads
        concat_output = self.combine_heads(attn_output)

        # Final linear transformation
        output = self.W_o(concat_output)

        return output, attn_weights


# Main Program

torch.manual_seed(42)

# Define parameters
batch_size = 1
seq_len = 3
d_model = 4
num_heads = 4

# Create random input embeddings
x = torch.randn(
    batch_size,
    seq_len,
    d_model
)

# Initialize Multi-Head Attention
mha = MultiHeadAttention(
    d_model=d_model,
    num_heads=num_heads
)


# TEST 1: Unmasked Self-Attention

output, attn_weights = mha(
    query=x,
    key=x,
    value=x
)

print("SELF-ATTENTION")
print("-----------------------------")

print("Input Shape:", x.shape)
print("Output Shape:", output.shape)

print("\nAttention Weights Shape:")
print(attn_weights.shape)

print("\nAttention Weights:")
print(attn_weights)

print("\nAttention Output:")
print(output)


# TEST 2: Causal Masking

causal_mask = torch.tril(
    torch.ones(seq_len, seq_len)
).unsqueeze(0).unsqueeze(0)

masked_output, masked_weights = mha(
    query=x,
    key=x,
    value=x,
    mask=causal_mask
)

print("\nCAUSAL MASKING")
print("-----------------------------")

print("Causal Mask:")
print(causal_mask)

print("\nInput Shape:", x.shape)
print("Number of Heads:", num_heads)

print("\nAttention Weights Shape:")
print(masked_weights.shape)

print("\nMulti-Head Attention Output:")
print(masked_output)

print("\nOutput Shape:")
print(masked_output.shape)

SELF-ATTENTION
-----------------------------
Input Shape: torch.Size([1, 3, 4])
Output Shape: torch.Size([1, 3, 4])

Attention Weights Shape:
torch.Size([1, 4, 3, 3])

Attention Weights:
tensor([[[[0.3375, 0.3314, 0.3312],
          [0.3178, 0.3407, 0.3416],
          [0.3470, 0.3269, 0.3262]],

         [[0.3070, 0.3712, 0.3219],
          [0.4978, 0.1398, 0.3624],
          [0.2811, 0.4099, 0.3089]],

         [[0.3034, 0.3726, 0.3241],
          [0.3959, 0.2589, 0.3453],
          [0.2887, 0.3925, 0.3187]],

         [[0.2858, 0.4382, 0.2760],
          [0.2854, 0.4390, 0.2756],
          [0.2867, 0.4362, 0.2771]]]], grad_fn=<SoftmaxBackward0>)

Attention Output:
tensor([[[-0.4398,  0.0742,  0.1407, -0.3353],
         [-0.3663,  0.0420, -0.0021, -0.3062],
         [-0.4527,  0.0801,  0.1638, -0.3408]]], grad_fn=<ViewBackward0>)

CAUSAL MASKING
-----------------------------
Causal Mask:
tensor([[[[1., 0., 0.],
          [1., 1., 0.],
          [1., 1., 1.]]]])

Input Shape: torch.Siz